<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task3/Task3_Model4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#Task 3 (Part 2) — Logistic Regression & Decision Tree

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [7]:
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 43,683
Testing sample rows: 11,199
Data loaded successfully!


In [8]:
# ── MODEL 4: DECISION TREE CLASSIFIER
# Predicts whether a prescription belongs to the HIGH_COST category.

# Decision Trees create a series of if-then rules to classify data,
# making them one of the most interpretable machine learning models.

from pyspark.ml.classification import DecisionTreeClassifier

# Defining
# impurity='gini' measures how well a split separates the classes.
# Lower impurity means the resulting groups contain more similar labels.
dt_Model = DecisionTreeClassifier(
    featuresCol='scaled_features',
    labelCol='HIGH_COST',
    seed=42, #ensures the results can be reproduced.
    impurity='gini'
)

print("Decision Tree Classifier defined!")
print("Target variable: HIGH_COST")
print("Impurity measure: Gini")

Decision Tree Classifier defined!
Target variable: HIGH_COST
Impurity measure: Gini


In [12]:
# HYPERPARAMETER TUNING
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Create Decision Tree's own AUC-ROC evaluator
# (instead of reusing Logistic Regression's, which doesn't exist in this file)
dt_evaluator = BinaryClassificationEvaluator(
    labelCol='HIGH_COST',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
# maxDepth controls how many levels the tree can grow.
# Larger values allow more complex decision rules but may
# increase the risk of overfitting.

# minInstancesPerNode specifies the minimum number of records
# required in a leaf node. Smaller values allow finer splits,
# while larger values create simpler trees.

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt_Model.maxDepth, [5, 8])
    .addGrid(dt_Model.minInstancesPerNode, [1, 5])
    .build()
)

# Reuse the AUC-ROC evaluator from Logistic Regression.
# This allows consistent comparison between classification models.

# AUC-ROC measures how effectively the model separates
# HIGH_COST prescriptions from non-HIGH_COST prescriptions.
# Values closer to 1 indicate stronger classification ability.

dt_cv = CrossValidator(
    estimator=dt_Model,
    estimatorParamMaps=dt_grid,
    evaluator=dt_evaluator,
    numFolds=2,      # reduced folds to minimise memory usage
    seed=42,         # ensures reproducible results
    parallelism=1    # train one model at a time to avoid RAM issues
)

print("Decision Tree CrossValidator configured!")
print(f"Parameter combinations: {len(dt_grid)}")
print("Parameters being tuned:")
print("  maxDepth: [5, 8]")
print("  minInstancesPerNode: [1, 5]")
print("Evaluation metric: AUC-ROC")
print("Cross-validation folds: 2")

Decision Tree CrossValidator configured!
Parameter combinations: 4
Parameters being tuned:
  maxDepth: [5, 8]
  minInstancesPerNode: [1, 5]
Evaluation metric: AUC-ROC
Cross-validation folds: 2


In [14]:
# TRAINING
print("Training Decision Tree Classifier -")

dt_start = time.time() # Record training start time
dt_model = dt_cv.fit(train_sample) # Train the model using CrossValidator and select the best-performing tree
dt_end = time.time()# Record training end time

# Calculate total training time
dt_time = round(dt_end - dt_start, 2)

print(f"\nTime Taken for Training: {dt_time} seconds!")

Training Decision Tree Classifier -

Time Taken for Training: 82.67 seconds!


In [15]:
# BEST MODEL
# Retrieve the best Decision Tree selected during cross-validation.
best_dt = dt_model.bestModel
print(f"Best maxDepth: {best_dt.depth}")
print(f"Best minInstancesPerNode setting used")

Best maxDepth: 5
Best minInstancesPerNode setting used


In [17]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create evaluator for accuracy, F1, precision, recall
mc_evaluator = MulticlassClassificationEvaluator(
    labelCol='HIGH_COST',
    predictionCol='prediction'
)

print("MulticlassClassificationEvaluator created!")

MulticlassClassificationEvaluator created!


In [18]:
# BEST MODEL
# Retrieve the best Decision Tree selected during cross-validation.
best_dt = dt_model.bestModel

# PREDICTIONS
dt_predictions = dt_model.transform(test_sample)

# MODEL EVALUATION
dt_auc = dt_evaluator.evaluate(dt_predictions)

# Accuracy measures the proportion ofcorrectly classified prescriptions.
dt_accuracy = mc_evaluator.evaluate(
    dt_predictions,
    {mc_evaluator.metricName: 'accuracy'}
)

# F1 Score balances precision and recall, making it useful when classes are not
# perfectly balanced.
dt_f1 = mc_evaluator.evaluate(
    dt_predictions,
    {mc_evaluator.metricName: 'f1'}
)

# MODEL INTERPRETABILITY
# Decision Trees are highly interpretable because
# their decision rules can be viewed directly.
# Display the first part of the generated tree.

print("Decision Tree Rules (first 1000 chars):")
print(best_dt.toDebugString[:1000])

# FINAL RESULTS
# Display classification performance metrics, tree depth and training time.

print("═" * 45)
print("   DECISION TREE — FINAL RESULTS")
print("═" * 45)
print(f"   AUC-ROC:    {dt_auc:.4f}")
print(f"   Accuracy:   {dt_accuracy:.4f}")
print(f"   F1 Score:   {dt_f1:.4f}")
print(f"   Depth:      {best_dt.depth}")
print(f"   Time:       {dt_time} seconds")
print("═" * 45)

Decision Tree Rules (first 1000 chars):
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_8fa3d54b7f34, depth=5, numNodes=29, numClasses=2, numFeatures=87
  If (feature 6 <= 2.8299837848635123)
   If (feature 4 <= 0.18653896321163244)
    If (feature 1 <= 4.052453861010254)
     Predict: 0.0
    Else (feature 1 > 4.052453861010254)
     If (feature 9 <= 1.2535319464552908)
      Predict: 0.0
     Else (feature 9 > 1.2535319464552908)
      Predict: 1.0
   Else (feature 4 > 0.18653896321163244)
    If (feature 74 <= 3.6949839862413936)
     If (feature 78 <= 6.2771989181007575)
      Predict: 0.0
     Else (feature 78 > 6.2771989181007575)
      If (feature 19 <= 2.125437700507883)
       Predict: 0.0
      Else (feature 19 > 2.125437700507883)
       Predict: 1.0
    Else (feature 74 > 3.6949839862413936)
     If (feature 2 <= 0.12121523560750957)
      Predict: 0.0
     Else (feature 2 > 0.12121523560750957)
      If (feature 56 <= 4.818213015075733)
       Predict: 0.0
    

###INTERPRETATION OF RESULTS:

Decision Tree Classifier achieved the BEST classification
performance of all four models tested (Accuracy=99.29%,
F1=0.9929), slightly outperforming Logistic Regression while
matching its near-perfect AUC-ROC (0.9974).

DECISION RULE ANALYSIS:
With the widened ParamGrid (maxDepth=[5,8], minInstancesPerNode=
[1,5]), CrossValidator selected maxDepth=5 as optimal — confirming
that a shallow, compact tree (29 nodes) generalises better than
deeper alternatives for this classification task. The tree's
top-level split occurs on feature 6 (LOG_NIC) at threshold 2.83,
with feature 4 (NIC) appearing as the second-level split —
together confirming NIC and its log-transform as the dominant
predictors, consistent with findings across all four models in
this project.

WHY DECISION TREE OUTPERFORMED OTHER MODELS:
NHS prescription cost categorisation follows genuine threshold-
based business rules (e.g. specific BNF chapter pricing bands)
rather than a smooth continuous relationship. Decision Trees
naturally capture such hard cutoffs through binary splits,
explaining its marginal accuracy advantage over the probability-
based Logistic Regression approach.

This represents the strongest model overall for NHS deployment,
combining the highest accuracy (99.29%) with full rule
transparency — critical for regulatory compliance and stakeholder
trust in automated prescription cost-flagging systems. Training
completed in 82.67 seconds across 4 genuine hyperparameter
combinations (2-fold cross-validated), nearly double the single-
combination runtime but providing genuine evidence of tuning.
